In [30]:
%pip install ipympl

     ------------------------------------ 519.0/519.0 kB 354.1 kB/s eta 0:00:00
     ------------------------------------- 139.8/139.8 kB 98.8 kB/s eta 0:00:00
     ---------------------------------------- 2.2/2.2 MB 84.5 kB/s eta 0:00:00
     ------------------------------------- 914.9/914.9 kB 76.9 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'c:\\Users\\Souherdya\\OneDrive\\Desktop\\Signal Processing IIT KGP\\Range_Doppler_AOA\\.venv\\share\\jupyter\\labextensions\\@jupyter-widgets\\jupyterlab-manager\\static\\vendors-node_modules_base64-js_index_js-node_modules_sanitize-html_index_js.c79fc2bcc3f69676beda.js.map'


[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
%matplotlib notebook

In [4]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# from configuration import *
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.widgets import Button
from openradar.mmwave.dataloader.adc import DCA1000
from openradar.mmwave import dsp
from openradar.mmwave.dsp.utils import Window
import time
from datetime import datetime
import sys

In [27]:
numFrames = 600
numADCSamples = 256
numTxAntennas = 3
numRxAntennas = 4
numLoopsPerFrame = 182
numChirpsPerFrame = numTxAntennas * numLoopsPerFrame

In [15]:
DATA_PATH="../data/adc_data_2026-02-06_16-35-07_land_vib_80.npy"

In [16]:

stop_flag = False
start_flag = True  
Tp = 14e-6
Tc = 72e-6

In [17]:

def init_dca():
    dca = DCA1000()
    return dca 

def collect_data(dca, num_frames=1):
    adc_data = dca.read(num_frames=int(num_frames))
    return adc_data

def stop_plot(event):
    global stop_flag, start_flag
    stop_flag = True
    start_flag = False

def start_plot(event):
    global stop_flag, start_flag
    stop_flag = False
    start_flag = True

In [18]:

def get_pcd(det_matrix):
    fft2d_sum = det_matrix.astype(np.int64)
    thresholdDoppler, noiseFloorDoppler = np.apply_along_axis(func1d=dsp.ca_,
                                                                axis=0,
                                                                arr=fft2d_sum.T,
                                                                l_bound=1.5,
                                                                guard_len=4,
                                                                noise_len=16)

    thresholdRange, noiseFloorRange = np.apply_along_axis(func1d=dsp.ca_,
                                                            axis=0,
                                                            arr=fft2d_sum,
                                                            l_bound=2.5,
                                                            guard_len=4,
                                                            noise_len=16)

    thresholdDoppler, noiseFloorDoppler = thresholdDoppler.T, noiseFloorDoppler.T
    det_doppler_mask = (det_matrix > thresholdDoppler)
    det_range_mask = (det_matrix > thresholdRange)

    # Get indices of detected peaks
    full_mask = (det_doppler_mask & det_range_mask)
    det_peaks_indices = np.argwhere(full_mask == True)

    # peakVals and SNR calculation
    peakVals = fft2d_sum[det_peaks_indices[:, 0], det_peaks_indices[:, 1]]
    snr = peakVals - noiseFloorRange[det_peaks_indices[:, 0], det_peaks_indices[:, 1]]

    dtype_location = '(' + str(numTxAntennas) + ',)<f4'
    dtype_detObj2D = np.dtype({'names': ['rangeIdx', 'dopplerIdx', 'peakVal', 'location', 'SNR'],
                                'formats': ['<i4', '<i4', '<f4', dtype_location, '<f4']})
    detObj2DRaw = np.zeros((det_peaks_indices.shape[0],), dtype=dtype_detObj2D)
    detObj2DRaw['rangeIdx'] = det_peaks_indices[:, 0].squeeze()
    detObj2DRaw['dopplerIdx'] = det_peaks_indices[:, 1].squeeze()
    detObj2DRaw['peakVal'] = peakVals.flatten()
    detObj2DRaw['SNR'] = snr.flatten()

    # Further peak pruning. This increases the point cloud density but helps avoid having too many detections around one object.
    detObj2DRaw = dsp.prune_to_peaks(detObj2DRaw, det_matrix, numDopplerBins, reserve_neighbor=True)

    # --- Peak Grouping
    detObj2D = dsp.peak_grouping_along_doppler(detObj2DRaw, det_matrix, numDopplerBins)
    SNRThresholds2 = np.array([[2, 23], [10, 11.5], [35, 16.0]])
    peakValThresholds2 = np.array([[4, 275], [1, 400], [500, 0]])
    detObj2D = dsp.range_based_pruning(detObj2D, SNRThresholds2, peakValThresholds2, numRangeBins, 0.5, range_resolution)

    azimuthInput = aoa_input[detObj2D['rangeIdx'], :, detObj2D['dopplerIdx']]

    Psi, Theta, Ranges, xyzVec = dsp.beamforming_naive_mixed_xyz(azimuthInput, detObj2D['rangeIdx'],
                                                                    range_resolution, method='Bartlett')
    return xyzVec

In [19]:

def iterative_range_bins_detection(rangeResult):
    rangeResult = np.transpose(np.stack([rangeResult[0::3], rangeResult[1::3], rangeResult[2::3]], axis=1),axes=(1,2,0,3))
    range_result_absnormal_split=[]
    for i in range(numTxAntennas):
        for j in range(numRxAntennas):
            r_r=np.abs(rangeResult[i][j])
            #first 10 range bins i.e 40 cm make it zero
            r_r[:,0:10]=0
            min_val = np.min(r_r)
            max_val = np.max(r_r)
            r_r_normalise = (r_r - min_val) / (max_val - min_val) * (1000 - 0) + 0
            range_result_absnormal_split.append(r_r_normalise)
    
    range_abs_combined_nparray=np.zeros((numLoopsPerFrame,numADCSamples))
    for ele in range_result_absnormal_split:
        range_abs_combined_nparray+=ele
    range_abs_combined_nparray/=(numTxAntennas*numRxAntennas)
    
    range_abs_combined_nparray_collapsed=np.sum(range_abs_combined_nparray,axis=0)/numLoopsPerFrame
    peaks_min_intensity_threshold = np.argsort(range_abs_combined_nparray_collapsed)[::-1][:5]
    max_range_index=np.argmax(range_abs_combined_nparray_collapsed)
    return max_range_index, peaks_min_intensity_threshold, rangeResult

In [20]:
def get_phase(r,i):
    if r==0:
        if i>0:
            phase=np.pi/2
        else :
            phase=3*np.pi/2
    elif r>0:
        if i>=0:
            phase=np.arctan(i/r)
        if i<0:
            phase=2*np.pi - np.arctan(-i/r)
    elif r<0:
        if i>=0:
            phase=np.pi - np.arctan(-i/r)
        else:
            phase=np.pi + np.arctan(i/r)
    return phase

In [21]:
def phase_unwrapping(phase_len,phase_cur_frame):
    i=1
    new_signal_phase = phase_cur_frame
    for k,ele in enumerate(new_signal_phase):
        if k==len(new_signal_phase)-1:
            continue
        if new_signal_phase[k+1] - new_signal_phase[k] > 1.5*np.pi:
            new_signal_phase[k+1:] = new_signal_phase[k+1:] - 2*np.pi*np.ones(len(new_signal_phase[k+1:]))
    return np.array(new_signal_phase)

In [22]:
def solve_equation(phase_cur_frame):
    phase_diff=[]
    for soham in range (1,len(phase_cur_frame)):
        phase_diff.append(phase_cur_frame[soham]-phase_cur_frame[soham-1])
    L=100
    r0=20
    roots_of_frame=[]
    for i,val in enumerate(phase_diff):
        c=(phase_diff[i]*0.001/3.14)/(3*(Tp+Tc))
        t=3*(i+1)*(Tp+Tc)
        c1=t*t
        c2=-2*L*t
        c3=L*L-c*c*t*t
        c4=2*L*c*c*t
        c5=-r0*r0*c*c
        coefficients=[c1, c2, c3, c4, c5]
        root=min(np.abs(np.roots(coefficients)))
        roots_of_frame.append(root)
    median_root=np.median(roots_of_frame)
    final_roots=[]
    for root in roots_of_frame:
        if root >0.9*median_root and root<1.1*median_root:
            final_roots.append(root)
    return np.mean(final_roots)

In [23]:
def get_velocity_antennawise(range_FFT_,peak):
    phase_per_antenna=[]
    vel_peak=[]
    for k in range(0,numLoopsPerFrame):
        r = range_FFT_[k][peak].real
        i = range_FFT_[k][peak].imag
        phase=get_phase(r,i)
        phase_per_antenna.append(phase)
    phase_cur_frame=phase_unwrapping(len(phase_per_antenna),phase_per_antenna)
    cur_vel=solve_equation(phase_cur_frame)
    return cur_vel

In [24]:
def get_velocity(rangeResult,range_peaks):
    vel_array_frame=[]
    for peak in range_peaks:
        vel_arr_all_ant=[]
        for i in range(0,numTxAntennas):
            for j in range(0,numRxAntennas):
                cur_velocity=get_velocity_antennawise(rangeResult[i][j],peak)
                vel_arr_all_ant.append(cur_velocity)
        vel_array_frame.append(vel_arr_all_ant)
    return vel_array_frame

In [25]:
def dopplerFFT(rangeResult):  #
    windowedBins2D = rangeResult * np.reshape(np.hamming(numLoopsPerFrame), (1, 1, -1, 1))
    dopplerFFTResult = np.fft.fft(windowedBins2D, axis=2)
    dopplerFFTResult = np.fft.fftshift(dopplerFFTResult, axes=2)
    return dopplerFFTResult


def speed_estimation_fn(range_bins, rangeResult):
    vel_array_frame = np.array(get_velocity(rangeResult,range_bins)).flatten()
    return vel_array_frame

In [33]:
if __name__ == "__main__":
    # dca = init_dca()
    plt.ion()
    fig = None
    i = 0 
    prev_range_bins = None
    overlapped_range_bins = []

    loaded_adc_data = np.load(DATA_PATH)
    while i < 100:
        # i+=1
        current_frame = loaded_adc_data[i:i+1]
        adc_data = np.apply_along_axis(DCA1000.organize, 1, current_frame, num_chirps=numChirpsPerFrame, num_rx=numRxAntennas, num_samples=numADCSamples)
        radar_cube = dsp.range_processing(adc_data[0], window_type_1d=Window.BLACKMAN)
        rangefft_out = np.abs(radar_cube).sum(axis=(0,1))
        det_matrix, aoa_input = dsp.doppler_processing(radar_cube, num_tx_antennas=3, clutter_removal_enabled=True, window_type_2d=Window.HAMMING)
        det_matrix_vis = np.fft.fftshift(det_matrix, axes=1)
        max_range_index, range_bins, rangeResult = iterative_range_bins_detection(radar_cube)
        if i < 5:
            overlapped_range_bins.append(range_bins)
            prev_range_bins = range_bins
        else:
            last_frame_idx = len(overlapped_range_bins)
            curr_ranges = set()
            for prev_range_bin in prev_range_bins:
                for cur_range_bin in range_bins:
                    if abs(prev_range_bin - cur_range_bin) <= 5:
                        #if within +/- 3, then keep the range bins 
                        curr_ranges.add(cur_range_bin)
            prev_range_bins = range_bins
            overlapped_range_bins.append(np.array(list(curr_ranges)))
            range_bins = overlapped_range_bins[-1]
        print(f"range_bins: {range_bins}, prev_range_bins: {prev_range_bins}, overlapped_range_bins: {overlapped_range_bins}")
        vel_array_frame = speed_estimation_fn(range_bins, rangeResult)
        print("vel_array_frame", vel_array_frame.shape)

        if fig is None:
            fig = plt.figure(figsize=(18, 10))
            ax1 = fig.add_subplot(1, 3, 1)
            ax2 = fig.add_subplot(1, 3, 2)
            ax3 = fig.add_subplot(1, 3, 3)

            ax_stop = fig.add_axes([0.75, 0.92, 0.1, 0.05])
            ax_start = fig.add_axes([0.87, 0.92, 0.1, 0.05])
            btn_stop = Button(ax_stop, 'Stop')
            btn_start = Button(ax_start, 'Start')
            btn_stop.on_clicked(stop_plot)
            btn_start.on_clicked(start_plot)

        for ax in [ax1, ax2, ax3]:
            ax.cla()

        ax1.plot(rangefft_out)
        ax1.set_title("RangeFFT")

        sns.heatmap(det_matrix_vis / det_matrix_vis.max(), ax=ax2, cbar=False, cmap='viridis')
        ax2.set_title("Range-doppler Heatmap")
        ax3.axis('off')
        message = f"Iteration {i+1}/100\nEstimated Speed: {vel_array_frame.mean():.2f}"
        ax3.text(0.5, 0.5, message, ha='center', va='center', fontsize=14, fontweight='bold')

        plt.tight_layout()
        plt.pause(0.1)

        i += 1  

    plt.ioff()
    plt.show()


range_bins: [255  16  17  15  14], prev_range_bins: [255  16  17  15  14], overlapped_range_bins: [array([255,  16,  17,  15,  14])]
vel_array_frame (60,)


<IPython.core.display.Javascript object>

C:\Users\Souherdya\AppData\Local\Temp\ipykernel_44732\2575140418.py:62: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


range_bins: [255  16  17  15  14], prev_range_bins: [255  16  17  15  14], overlapped_range_bins: [array([255,  16,  17,  15,  14]), array([255,  16,  17,  15,  14])]
vel_array_frame (60,)


C:\Users\Souherdya\AppData\Local\Temp\ipykernel_44732\2575140418.py:62: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


...

C:\Users\Souherdya\AppData\Local\Temp\ipykernel_44732\2575140418.py:62: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


range_bins: [ 13  15  16  17 255], prev_range_bins: [255  16  17  13  15], overlapped_range_bins: [array([255,  16,  17,  15,  14]), array([255,  16,  17,  15,  14]), array([255,  16,  17,  11,  26]), array([255,  16,  17,  13,  15]), array([255,  16,  17,  15,  12]), array([ 16,  17,  15, 255]), array([ 15,  16,  17,  26, 255]), array([ 12,  15,  16,  17, 255]), array([ 16,  17,  15, 255]), array([ 15,  16,  17,  26, 255]), array([ 14,  15,  16,  17, 255]), array([ 12,  15,  16,  17, 255]), array([ 14,  15,  16,  17, 255]), array([ 13,  14,  16,  17, 255]), array([ 13,  14,  16,  17, 255]), array([ 13,  15,  16,  17, 255]), array([ 12,  13,  16,  17, 255]), array([ 12,  13,  16,  17, 255]), array([ 14,  15,  16,  17, 255]), array([ 13,  15,  16,  17, 255]), array([ 13,  15,  16,  17, 255]), array([ 12,  13,  16,  17, 255]), array([ 11,  15,  16,  17, 255]), array([ 14,  15,  16,  17, 255]), array([ 14,  15,  16,  17, 255]), array([ 14,  15,  16,  17, 255]), array([ 14,  15,  16,  17, 

C:\Users\Souherdya\AppData\Local\Temp\ipykernel_44732\2575140418.py:62: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


range_bins: [ 13  15  16  17 255], prev_range_bins: [255  16  17  15  13], overlapped_range_bins: [array([255,  16,  17,  15,  14]), array([255,  16,  17,  15,  14]), array([255,  16,  17,  11,  26]), array([255,  16,  17,  13,  15]), array([255,  16,  17,  15,  12]), array([ 16,  17,  15, 255]), array([ 15,  16,  17,  26, 255]), array([ 12,  15,  16,  17, 255]), array([ 16,  17,  15, 255]), array([ 15,  16,  17,  26, 255]), array([ 14,  15,  16,  17, 255]), array([ 12,  15,  16,  17, 255]), array([ 14,  15,  16,  17, 255]), array([ 13,  14,  16,  17, 255]), array([ 13,  14,  16,  17, 255]), array([ 13,  15,  16,  17, 255]), array([ 12,  13,  16,  17, 255]), array([ 12,  13,  16,  17, 255]), array([ 14,  15,  16,  17, 255]), array([ 13,  15,  16,  17, 255]), array([ 13,  15,  16,  17, 255]), array([ 12,  13,  16,  17, 255]), array([ 11,  15,  16,  17, 255]), array([ 14,  15,  16,  17, 255]), array([ 14,  15,  16,  17, 255]), array([ 14,  15,  16,  17, 255]), array([ 14,  15,  16,  17, 

KeyboardInterrupt: 